In [33]:
import torch

In [34]:
import pandas as pd
filePath = "/Users/tharcisse/Desktop/AdamResearch/Codes/free-range-zoo/free_range_zoo/envs/wildfire/configs/outputs/wildfire_logging_test_0/done.csv"
data = pd.read_csv(filePath)

In [35]:
import ast
def safe_literal_eval(val):
    if pd.isna(val) or val == 'NULL':
        return None
    try:
        return ast.literal_eval(val)
    except:
        return val

In [36]:
columns = ['fires', 'intensity', 'fuel', 'suppressants', 'capacity', 
                   'equipment', 'agents', 'firefighter_1_action', 'firefighter_2_action', 
                   'firefighter_3_action']

for col in columns:
    if col in data.columns:
        data[col] = data[col].apply(safe_literal_eval)

In [37]:
#Find fire positions at each step
fires = data['fires']
fires_map = {}
agents_map = {}

for idx, grid in enumerate(fires):
    if not grid:
        fires_map[idx] = []
        agents_map[idx] = []
        continue
    fire_coords = []
    agent_coords = []
    for i, row in enumerate(grid):
        for j,cell in enumerate(row):
            if cell > 0:
                fire_coords.append((i,j))
            elif cell == 0:
                # This is an agent
                agent_coords.append((i,j))
    fires_map[idx] = fire_coords
    agents_map[idx] = agent_coords
print(fires_map)
print(agents_map)

{0: [(0, 1), (2, 1)], 1: [(0, 1), (2, 1)], 2: [(2, 1)], 3: [(2, 1)], 4: [(2, 1)], 5: [(2, 1)], 6: [(2, 1)], 7: [(2, 0), (2, 1)], 8: [(0, 2), (2, 0), (2, 1)], 9: [(0, 2), (2, 0), (2, 1)], 10: [(0, 2), (2, 0), (2, 1)], 11: [(0, 2), (2, 1)], 12: [(0, 2)], 13: [(0, 1), (0, 2)], 14: [(0, 1)], 15: []}
{0: [(0, 0), (1, 1), (2, 2)], 1: [(0, 0), (1, 1), (2, 2)], 2: [(0, 0), (1, 1), (2, 2)], 3: [(0, 0), (1, 1), (2, 2)], 4: [(0, 0), (1, 1), (2, 2)], 5: [(0, 0), (1, 1), (2, 2)], 6: [(0, 0), (1, 1), (2, 2)], 7: [(0, 0), (1, 1), (2, 2)], 8: [(0, 0), (1, 1), (2, 2)], 9: [(0, 0), (1, 1), (2, 2)], 10: [(0, 0), (1, 1), (2, 2)], 11: [(0, 0), (1, 1), (2, 2)], 12: [(0, 0), (1, 1), (2, 2)], 13: [(0, 0), (1, 1), (2, 2)], 14: [(0, 0), (1, 1), (2, 2)], 15: [(0, 0), (1, 1), (2, 2)]}


In [38]:
def swap_coordinates(coords):
    return [(x,y) for (y,x) in coords]

In [ ]:
for step, agents in enumerate(data['agents']):
    fires = fires_map[step]
    print(f"we have {len(fires)} fires, {swap_coordinates(fires)}")
    for idx, agent_pos in enumerate(agents):
        fighter = f'firefighter_{idx+1}_action'
        if fighter in data.columns:
            action = data[fighter].iloc[step]
            if action is not None and isinstance(action, list) and len(action) == 2:
                fire_number, power = action
                if fire_number >= 0 and power >= 0:
                    fire_pos = fires_map[step]
                    if fire_pos and 0 <= fire_number < len(fire_pos):
                        pos = swap_coordinates(fire_pos)[fire_number]
                        print(f"Step {step}, Agent {idx+1}: Position {agent_pos}, Fighting fire {fire_number} at {pos}")
                    else:
                        print(f"Step {step}, Agent {idx+1}: Position {agent_pos}, Fighting fire {fire_number} at INVALID")
                if power < 0:
                    print(f"Step {step}, Agent {idx+1}: Position {agent_pos}, Fighting as NOOP")

    print()

we have 2 fires, [(1, 0), (1, 2)]

we have 2 fires, [(1, 0), (1, 2)]
Step 1, Agent 1: Position [0, 0], Fighting fire 0 at (1, 0)
Step 1, Agent 2: Position [1, 1], Fighting fire 0 at (1, 0)
Step 1, Agent 3: Position [2, 2], Fighting fire 0 at (1, 0)

we have 1 fires, [(1, 2)]
Step 2, Agent 1: Position [0, 0], Fighting fire 0 at (1, 2)
Step 2, Agent 2: Position [1, 1], Fighting fire 0 at (1, 2)
Step 2, Agent 3: Position [2, 2], Fighting fire 0 at (1, 2)

we have 1 fires, [(1, 2)]
Step 3, Agent 1: Position [0, 0], Fighting as NOOP
Step 3, Agent 2: Position [1, 1], Fighting as NOOP
Step 3, Agent 3: Position [2, 2], Fighting as NOOP

we have 1 fires, [(1, 2)]
Step 4, Agent 1: Position [0, 0], Fighting as NOOP
Step 4, Agent 2: Position [1, 1], Fighting fire 0 at (1, 2)
Step 4, Agent 3: Position [2, 2], Fighting fire 0 at (1, 2)

we have 1 fires, [(1, 2)]
Step 5, Agent 1: Position [0, 0], Fighting as NOOP
Step 5, Agent 2: Position [1, 1], Fighting fire 0 at (1, 2)
Step 5, Agent 3: Position [2